# Layer-1 attention activation trajectories

This notebook follows one seeded example through the middle Transformer layer (layer 1). It plots the residual-stream trajectory before attention (`blocks.1.hook_resid_pre`) and after attention but before the MLP (`blocks.1.hook_resid_mid`). It then repeats the post-attention trajectory after independently mean-ablating each layer-1 attention head.

All trajectories use one shared three-dimensional PCA basis, fit to the clean and ablated activations together, so their geometry is directly comparable.

In [48]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from IPython.display import display
from sklearn.decomposition import PCA

# Locate the project robustly when this notebook is opened from the project
# root, notebooks/, or this notebook's subfolder.
PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    if (candidate / "results/line_breaks_results").exists()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = PROJECT_ROOT / "results/line_breaks_results"
CHECKPOINT_PATH = RUN_DIR / "checkpoints/final.pt"

LAYER = 1
EXAMPLE_SEED = 27_005
CALIBRATION_SEED = 27_002
N_CALIBRATION_SEQUENCES = 512
CALIBRATION_BATCH_SIZE = 64
PCA_SEED = 27_003


In [49]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(
    path=CHECKPOINT_PATH, model=model, map_location=device
)
model.eval()

n_heads = cfg["model"]["n_heads"]
d_head = cfg["model"]["d_head"]
seq_len = cfg["task"]["seq_len"]
PRE_SITE = f"blocks.{LAYER}.hook_resid_pre"
POST_SITE = f"blocks.{LAYER}.hook_resid_mid"
Z_SITE = f"blocks.{LAYER}.attn.hook_z"

print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Layer {LAYER}: {n_heads} heads, d_head={d_head}")
print("Before-attention site:", PRE_SITE)
print("After-attention site:", POST_SITE)


Building vocab...
('BOS', '_NEWLINE_')
Loaded checkpoint step 10000 on cpu
Layer 1: 4 heads, d_head=32
Before-attention site: blocks.1.hook_resid_pre
After-attention site: blocks.1.hook_resid_mid


## Estimate mean-ablation vectors for layer-1 heads

In [50]:
@torch.no_grad()
def estimate_head_means():
    sums = torch.zeros(n_heads, d_head, dtype=torch.float64)
    n_vectors = 0
    for start in range(0, N_CALIBRATION_SEQUENCES, CALIBRATION_BATCH_SIZE):
        size = min(CALIBRATION_BATCH_SIZE, N_CALIBRATION_SEQUENCES - start)
        batch = project_data.make_batch(
            batch_size=size, vocab=vocab, task_cfg=cfg["task"],
            device=device, seed=CALIBRATION_SEED + start,
        )
        _, cache = model.run_with_cache(
            batch.tokens, prepend_bos=False, names_filter=[Z_SITE]
        )
        sums += cache[Z_SITE].detach().cpu().double().sum(dim=(0, 1))
        n_vectors += size * seq_len
    return (sums / n_vectors).float()

replacement_head_outputs = estimate_head_means()
print("Mean replacement norms by head:", replacement_head_outputs.norm(dim=-1).numpy())


Mean replacement norms by head: [2.5261836 4.0299153 3.1272478 3.1672242]


## Collect the clean and independently ablated trajectories

In [51]:
def character_counts(token_ids):
    counts = []
    current = 0
    for token_id in token_ids:
        counts.append(current)
        token_name = vocab.decode_token(int(token_id))
        if token_name == "_NEWLINE_":
            current = 0
        else:
            current += project_data.extract_character_count(token_name)
    return np.asarray(counts)

@torch.no_grad()
def collect_trajectory(fwd_hooks=None):
    batch = project_data.make_batch(
        batch_size=1, vocab=vocab, task_cfg=cfg["task"],
        device=device, seed=EXAMPLE_SEED,
    )
    if not fwd_hooks:
        _, cache = model.run_with_cache(
            batch.tokens, prepend_bos=False, names_filter=[PRE_SITE, POST_SITE, Z_SITE]
        )
        pre_activation = cache[PRE_SITE][0].detach().cpu().numpy()
        post_activation = cache[POST_SITE][0].detach().cpu().numpy()
        head_output = cache[Z_SITE][0].detach().cpu().numpy()
    else:
        captured = {}
        def capture_post(activation, hook):
            captured["post"] = activation.detach()
            return activation
        hooked = list(fwd_hooks) + [(POST_SITE, capture_post)]
        model.run_with_hooks(batch.tokens, prepend_bos=False, fwd_hooks=hooked)
        pre_activation = pre_clean
        post_activation = captured["post"][0].cpu().numpy()
        head_output = None
    return (
        batch.tokens[0].detach().cpu().numpy(),
        batch.line_lengths[0].item(),
        pre_activation,
        post_activation,
        head_output,
    )

tokens, line_length, pre_clean, post_clean, clean_head_outputs = collect_trajectory()
counts = character_counts(tokens)

ablated_post = {}
for head in range(n_heads):
    replacement = replacement_head_outputs[head]
    def ablate_head(z, hook, head=head, replacement=replacement):
        z_ablated = z.clone()
        z_ablated[:, :, head, :] = replacement.to(device=z.device, dtype=z.dtype)
        return z_ablated
    _, _, _, post, _ = collect_trajectory(fwd_hooks=[(Z_SITE, ablate_head)])
    ablated_post[head] = post

zero_ablated_post = {}
for head in range(n_heads):
    def zero_ablate_head(z, hook, head=head):
        z_ablated = z.clone()
        z_ablated[:, :, head, :] = 0.0
        return z_ablated
    _, _, _, post, _ = collect_trajectory(fwd_hooks=[(Z_SITE, zero_ablate_head)])
    zero_ablated_post[head] = post

KEEP_HEAD = 1
def zero_other_heads(z, hook):
    z_ablated = z.clone()
    for head in range(n_heads):
        if head != KEEP_HEAD:
            z_ablated[:, :, head, :] = 0.0
    return z_ablated
_, _, _, zero_except_head_1_post, _ = collect_trajectory(fwd_hooks=[(Z_SITE, zero_other_heads)])

print("Example line length:", line_length)
print("Trajectory shapes:", pre_clean.shape, post_clean.shape)
print("Mean-ablated heads:", sorted(ablated_post))
print("Zero-ablated heads:", sorted(zero_ablated_post))


Example line length: 58
Trajectory shapes: (150, 128) (150, 128)
Mean-ablated heads: [0, 1, 2, 3]
Zero-ablated heads: [0, 1, 2, 3]


## Shared PCA coordinates

In [52]:
all_vectors = np.concatenate(
    [pre_clean, post_clean]
    + [ablated_post[head] for head in range(n_heads)]
    + [zero_ablated_post[head] for head in range(n_heads)],
    axis=0,
)
shared_pca = PCA(n_components=3, svd_solver="randomized", random_state=PCA_SEED)
shared_pca.fit(all_vectors)

pre_coords = shared_pca.transform(pre_clean)
post_coords = shared_pca.transform(post_clean)
ablated_coords = {head: shared_pca.transform(values) for head, values in ablated_post.items()}
zero_ablated_coords = {head: shared_pca.transform(values) for head, values in zero_ablated_post.items()}
zero_except_head_1_coords = shared_pca.transform(zero_except_head_1_post)

customdata = np.column_stack([
    np.arange(seq_len),
    counts,
    [vocab.decode_token(int(token_id)) for token_id in tokens],
])
print(f"Shared PCA variance explained: {shared_pca.explained_variance_ratio_.sum():.1%}")


Shared PCA variance explained: 88.1%


## How much does each head change the residual stream?

The PCA variance column is computed by fitting a separate three-dimensional PCA to each post-attention trajectory. It measures how much of that trajectory's own variance is captured by three dimensions; the trajectory plots below still use the shared PCA basis.

In [53]:
def trajectory_pca3_variance_explained(trajectory):
    trajectory_pca = PCA(n_components=3, svd_solver="randomized", random_state=PCA_SEED)
    trajectory_pca.fit(trajectory)
    return float(trajectory_pca.explained_variance_ratio_.sum())

ablation_mse_rows = [{
    "head": "clean",
    "post_attention_activation_MSE": 0.0,
    "post_attention_activation_RMSE": 0.0,
    "mean_euclidean_change": 0.0,
    "max_euclidean_change": 0.0,
    "post_trajectory_PCA3_variance_explained": trajectory_pca3_variance_explained(post_clean),
}]
for head, corrupted in ablated_post.items():
    difference = corrupted - post_clean
    ablation_mse_rows.append({
        "head": head,
        "post_attention_activation_MSE": float(np.mean(difference ** 2)),
        "post_attention_activation_RMSE": float(np.sqrt(np.mean(difference ** 2))),
        "mean_euclidean_change": float(np.linalg.norm(difference, axis=1).mean()),
        "max_euclidean_change": float(np.linalg.norm(difference, axis=1).max()),
        "post_trajectory_PCA3_variance_explained": trajectory_pca3_variance_explained(corrupted),
    })
ablation_mse_results = pd.DataFrame(ablation_mse_rows)
display(ablation_mse_results.style.format({
    "post_attention_activation_MSE": "{:.6f}",
    "post_attention_activation_RMSE": "{:.6f}",
    "mean_euclidean_change": "{:.4f}",
    "max_euclidean_change": "{:.4f}",
    "post_trajectory_PCA3_variance_explained": "{:.2%}",
}))


,head,post_attention_activation_MSE,post_attention_activation_RMSE,mean_euclidean_change,max_euclidean_change,post_trajectory_PCA3_variance_explained
0,clean,0.000000,0.000000,0.0000,0.0000,94.42%
1,0,0.204833,0.452585,5.0801,7.3037,94.47%
2,1,0.419235,0.647484,6.4824,14.6600,93.70%
3,2,0.063114,0.251225,2.5718,7.2657,94.46%
4,3,0.066855,0.258564,2.6818,4.6435,94.19%


## Similarity of head 2 and head 3 outputs

In [54]:
head_2_output = clean_head_outputs[:, 2, :]
head_3_output = clean_head_outputs[:, 3, :]
pairwise_difference = head_2_output - head_3_output
pairwise_distance = np.linalg.norm(pairwise_difference, axis=1)
head_2_norm = np.linalg.norm(head_2_output, axis=1)
head_3_norm = np.linalg.norm(head_3_output, axis=1)
cosine_similarity = np.sum(head_2_output * head_3_output, axis=1) / np.maximum(head_2_norm * head_3_norm, 1e-12)
relative_distance = pairwise_distance / np.maximum((head_2_norm + head_3_norm) / 2.0, 1e-12)
head_2_head_3_similarity = pd.DataFrame([{
    "mean_euclidean_distance": pairwise_distance.mean(),
    "median_euclidean_distance": np.median(pairwise_distance),
    "MSE_per_output_dimension": np.mean(pairwise_difference ** 2),
    "mean_cosine_similarity": cosine_similarity.mean(),
    "mean_relative_distance": relative_distance.mean(),
}])
display(head_2_head_3_similarity.style.format({
    "mean_euclidean_distance": "{:.6f}",
    "median_euclidean_distance": "{:.6f}",
    "MSE_per_output_dimension": "{:.6f}",
    "mean_cosine_similarity": "{:.4f}",
    "mean_relative_distance": "{:.2%}",
}))


,mean_euclidean_distance,median_euclidean_distance,MSE_per_output_dimension,mean_cosine_similarity,mean_relative_distance
0,4.088719,3.381136,0.579756,0.2958,119.37%


The Euclidean metrics compare head 2 and head 3 at the same token position. The relative distance divides by the mean norm of the two outputs at each position, which helps distinguish a large absolute difference caused simply by large output magnitudes from a genuinely different direction or trajectory.

## Raw output trajectory of each attention head

In [55]:
# Each plot uses a separate PCA fitted to that head's raw z output, which
# lives in d_head dimensions rather than residual-stream d_model space.
for head in range(n_heads):
    head_pca = PCA(n_components=3, svd_solver="randomized", random_state=PCA_SEED + head)
    head_coords = head_pca.fit_transform(clean_head_outputs[:, head, :])
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers",
        marker=dict(size=8, color="black"), name="Origin"))
    fig.add_trace(go.Scatter3d(
        x=head_coords[:, 0], y=head_coords[:, 1], z=head_coords[:, 2],
        mode="lines+markers", name=f"Head {head} output",
        marker=dict(size=4, color=counts, colorscale="Viridis", colorbar=dict(title="Character count")),
        line=dict(color="darkorange", width=5), customdata=customdata,
        hovertemplate="position=%{customdata[0]}<br>count=%{customdata[1]}<br>token=%{customdata[2]}<extra>head output</extra>",
    ))
    fig.update_layout(
        title=f"Raw layer {LAYER} head {head} output (hook_z); PCA variance explained = {head_pca.explained_variance_ratio_.sum():.1%}",
        scene=dict(xaxis_title="head-output PC1", yaxis_title="head-output PC2", zaxis_title="head-output PC3"),
        height=750,
    )
    display(fig)


## Clean trajectory: before and after layer-1 attention

In [56]:
def add_trajectory(fig, coords, name, color, *, mode="lines+markers", width=4):
    fig.add_trace(go.Scatter3d(
        x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
        mode=mode, name=name,
        marker=dict(size=4, color=counts, colorscale="Viridis",
                     colorbar=dict(title="Character count")),
        line=dict(color=color, width=width),
        customdata=customdata,
        hovertemplate="position=%{customdata[0]}<br>count=%{customdata[1]}<br>token=%{customdata[2]}<extra>" + name + "</extra>",
    ))

clean_fig = go.Figure()
clean_fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers",
    marker=dict(size=8, color="black"), name="Origin"))
add_trajectory(clean_fig, pre_coords, "Before attention", "royalblue")
add_trajectory(clean_fig, post_coords, "After attention", "crimson")
clean_fig.update_layout(
    title=f"Example seed {EXAMPLE_SEED}; layer {LAYER}, line length {line_length}",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    height=800,
)
display(clean_fig)


## One separate plot for each layer-1 head ablation

In [57]:
for head in range(n_heads):
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers",
        marker=dict(size=8, color="black"), name="Origin"))
    add_trajectory(fig, pre_coords, "Before attention", "royalblue", mode="lines", width=3)
    add_trajectory(fig, post_coords, "Clean after attention", "black", mode="lines", width=4)
    add_trajectory(fig, ablated_coords[head], f"After attention, head {head} mean-ablated", "darkorange", mode="lines+markers", width=5)
    fig.update_layout(
        title=f"Layer {LAYER}, head {head} ablation; shared PCA coordinates",
        scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
        height=800,
    )
    display(fig)


## Individual head zero-ablations

In [58]:
for head in range(n_heads):
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers",
        marker=dict(size=8, color="black"), name="Origin"))
    add_trajectory(fig, pre_coords, "Before attention", "royalblue", mode="lines", width=3)
    add_trajectory(fig, post_coords, "Clean after attention", "black", mode="lines", width=4)
    add_trajectory(fig, zero_ablated_coords[head], f"After attention, head {head} zero-ablated", "hotpink", mode="lines+markers", width=5)
    fig.update_layout(
        title=f"Layer {LAYER}, head {head} zero ablation; shared PCA coordinates",
        scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
        height=800,
    )
    display(fig)


## Keep head 1, zero-ablating all other layer-1 heads

In [59]:
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers",
    marker=dict(size=8, color="black"), name="Origin"))
add_trajectory(fig, pre_coords, "Before attention", "royalblue", mode="lines", width=3)
add_trajectory(fig, post_coords, "Clean after attention", "black", mode="lines", width=4)
add_trajectory(fig, zero_except_head_1_coords, "Head 1 intact; heads 0, 2, 3 zero-ablated", "limegreen", mode="lines+markers", width=6)
fig.update_layout(
    title="Layer 1: keep head 1 while zero-ablating all other heads",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    height=800,
)
display(fig)
